In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212

In [ ]:
!pip install kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting
!unzip -q walmart-recruiting-store-sales-forecasting.zip

100% 2.70M/2.70M [00:00<00:00, 219MB/s]



In [ ]:
!unzip -q train.csv.zip
!unzip -q stores.csv.zip
!unzip -q test.csv.zip
!unzip -q features.csv.zip

unzip:  cannot find or open stores.csv.zip, stores.csv.zip.zip or stores.csv.zip.ZIP.


In [ ]:
import mlflow
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error

from lightgbm import LGBMRegressor

In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner='tsarc21', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=b8368295-4b4c-4fb0-8e96-e8d9013015a1&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c81f84769996fa94dfa125eeba6d632041c84cfd10ef46e899ea8721c1efa881




Accessing as tsarc21

Initialized MLflow to track repo "tsarc21/Walmart-Recruiting---Store-Sales-Forecasting"

Repository tsarc21/Walmart-Recruiting---Store-Sales-Forecasting initialized!

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin


train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
stores = pd.read_csv("stores.csv")
features = pd.read_csv("features.csv")

merged_train = (
    train
    .merge(stores, on="Store", how="left")
    .merge(features, on=["Store", "Date", "IsHoliday"], how="left")
)

merged_test = (
    test
    .merge(stores, on="Store", how="left")
    .merge(features, on=["Store", "Date", "IsHoliday"], how="left")
)

merged_train["IsTest"] = 0
merged_test["IsTest"] = 1
merged_test["Weekly_Sales"] = np.nan

combined = pd.concat([merged_train, merged_test], ignore_index=True)

combined["Date"] = pd.to_datetime(combined["Date"])
combined = combined.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
combined = combined.replace([np.inf, -np.inf], np.nan)



class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.min_date_ = pd.to_datetime(X["Date"]).min()
        return self

    def transform(self, X):
        X = X.copy()
        X["Date"] = pd.to_datetime(X["Date"])

        X["Lag_52"] = (
            X.groupby(["Store", "Dept"], observed=True)["Weekly_Sales"]
            .shift(52) if "Weekly_Sales" in X.columns else None
        )

        if "Lag_52" in X.columns:
            X["Holiday_Lag52"] = X["IsHoliday"].astype(int) * X["Lag_52"]
        else:
            X["Holiday_Lag52"] = 0

        X["Year"] = X["Date"].dt.year
        X["Month"] = X["Date"].dt.month
        X["Week"] = X["Date"].dt.isocalendar().week.astype(int)
        X["Quarter"] = X["Date"].dt.quarter
        X["Days_from_start"] = (X["Date"] - self.min_date_).dt.days

        return X


class TargetNormalizer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        temp_df = X.copy()
        temp_df["Weekly_Sales"] = y

        holiday_stats = temp_df.groupby(["Store", "Dept", "IsHoliday"])["Weekly_Sales"].mean().unstack(fill_value=0)
        non_holiday_mean = holiday_stats[0] if 0 in holiday_stats.columns else pd.Series(1.0, index=holiday_stats.index)
        holiday_mean = holiday_stats[1] if 1 in holiday_stats.columns else pd.Series(1.0, index=holiday_stats.index)

        lift = holiday_mean / np.maximum(non_holiday_mean, 1e-5)
        self.lift_dict_ = lift.replace([np.inf, np.nan], 1.4).clip(lower=1.1, upper=2.5).to_dict()
        return self

    def transform(self, X):
        return X.copy()


class MarkdownFiller(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
        for col in markdown_cols:
            if col in X.columns:
                X[col] = X[col].fillna(0)
        return X


class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        cols_present = [col for col in self.columns_to_drop if col in X.columns]
        return X.drop(columns=cols_present)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            make_column_selector(dtype_include=["object", "category", "bool"])
        ),
        (
            "numerical",
            SimpleImputer(strategy="median"),
            make_column_selector(dtype_include=np.number)
        )
    ]
)



fe = FeatureEngineer()
combined = fe.fit_transform(combined)

train_full = combined[combined["IsTest"] == 0].copy()
test_final = combined[combined["IsTest"] == 1].copy()

train_full = train_full.dropna(subset=["Lag_52"]).copy()

y_full = train_full.pop("Weekly_Sales")
X_full = train_full.drop(columns=["IsTest"])
X_test = test_final.drop(columns=["IsTest", "Weekly_Sales"])



normalizer = TargetNormalizer()
normalizer.fit(X_full, y_full)

full_lifts = list(zip(X_full["Store"], X_full["Dept"]))
full_lift_series = pd.Series(full_lifts).map(normalizer.lift_dict_).fillna(1.4).values

y_full_base = np.where(
    X_full["IsHoliday"] == 1,
    y_full / full_lift_series,
    y_full
)



mlflow.set_experiment("Walmart_LGBM_Pipeline")

with mlflow.start_run(run_name="LGBM_Full_Pipeline"):

    params = {
        "objective": "regression",
        "random_state": 42,
        "n_jobs": -1,
        "subsample": 0.6,
        "reg_lambda": 10,
        "reg_alpha": 0,
        "num_leaves": 31,
        "n_estimators": 300,
        "min_child_samples": 20,
        "max_depth": 7,
        "learning_rate": 0.03,
        "colsample_bytree": 1.0
    }

    mlflow.log_params(params)

    pipeline = Pipeline(
        steps=[
            ("markdown_fill", MarkdownFiller()),
            ("drop_unnecessary_cols", DropColumns(["Date", "Weekly_Sales"])),
            ("preprocessor", preprocessor),
            ("model", LGBMRegressor(**params))
        ]
    )
    pipeline.fit(X_full, y_full_base)

    mlflow.sklearn.log_model(
        pipeline,
        "lgbm_pipeline_model",
        skops_trusted_types=[
            "__main__.DropColumns",
            "__main__.MarkdownFiller",
            "collections.OrderedDict",
            "lightgbm.basic.Booster",
            "lightgbm.sklearn.LGBMRegressor",
            "numpy.dtype",
            "numpy.number",
            "sklearn.compose._column_transformer.make_column_selector"
        ]
    )

    test_preds_base = pipeline.predict(X_test)

    test_lifts = list(zip(X_test["Store"], X_test["Dept"]))
    test_lift_series = pd.Series(test_lifts).map(normalizer.lift_dict_).fillna(1.4).values

    final_test_preds = np.where(
        X_test["IsHoliday"] == 1,
        test_preds_base * test_lift_series,
        test_preds_base
    )



submission = pd.DataFrame({
    "Store": X_test["Store"],
    "Dept": X_test["Dept"],
    "Date": X_test["Date"].dt.strftime("%Y-%m-%d"),
    "Weekly_Sales": final_test_preds
})

submission["Id"] = submission["Store"].astype(str) + "_" + submission["Dept"].astype(str) + "_" + submission["Date"].astype(str)
submission = submission[["Id", "Weekly_Sales"]]

submission.to_csv("submission.csv", index=False)

print("=" * 60)
print("SUCCESS: Model safely saved to MLflow & submission.csv generated!")
print(submission.head(10))
print("=" * 60)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.053778 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3123
[LightGBM] [Info] Number of data points in the train set: 261083, number of used features: 24
[LightGBM] [Info] Start training from score 16289.576919


2026/07/25 12:05:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 12:06:00 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmptrwbjqxr/model/model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.6.1', 'skops==0.14.0']. Set logging level to DEBUG to see the full traceback. 
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


🏃 View run LGBM_Full_Pipeline at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/24/runs/3546b2f4a5a440fc9a9f594ed76ae420
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/24
SUCCESS: Model safely saved to MLflow & submission.csv generated!
                 Id  Weekly_Sales
143  1_1_2012-11-02  39126.459195
144  1_1_2012-11-09  18519.058094
145  1_1_2012-11-16  20253.532525
146  1_1_2012-11-23  22309.452743
147  1_1_2012-11-30  26584.450131
148  1_1_2012-12-07  35141.206151
149  1_1_2012-12-14  45770.165418
150  1_1_2012-12-21  46356.944159
151  1_1_2012-12-28  27895.371955
152  1_1_2013-01-04  14395.106971


In [ ]:
import numpy as np
import pandas as pd


train = pd.read_csv("train.csv")
stores = pd.read_csv("stores.csv")
features = pd.read_csv("features.csv")



merged_data = (
    train
    .merge(
        stores,
        on="Store",
        how="left"
    )
    .merge(
        features,
        on=["Store", "Date", "IsHoliday"],
        how="left"
    )
)

merged_data["Date"] = pd.to_datetime(
    merged_data["Date"]
)

merged_data = merged_data.sort_values(
    ["Store", "Dept", "Date"]
)



merged_data["Lag_52"] = (
    merged_data
    .groupby(
        ["Store", "Dept"],
        observed=True
    )["Weekly_Sales"]
    .shift(52)
)



merged_data["Holiday_Lag52"] = (
    merged_data["IsHoliday"].astype(int)
    *
    merged_data["Lag_52"]
)



merged_data["Year"] = (
    merged_data["Date"].dt.year
)

merged_data["Month"] = (
    merged_data["Date"].dt.month
)

merged_data["Week"] = (
    merged_data["Date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

merged_data["Quarter"] = (
    merged_data["Date"].dt.quarter
)

merged_data["Days_from_start"] = (
    merged_data["Date"]
    -
    merged_data["Date"].min()
).dt.days


merged_data = merged_data.replace(
    [np.inf, -np.inf],
    np.nan
)

In [ ]:
split_date = "2011-08-01"

train_df = merged_data[
    merged_data["Date"] < split_date
].copy()

val_df = merged_data[
    merged_data["Date"] >= split_date
].copy()

train_df = train_df.dropna(
    subset=["Lag_52"]
)

val_df = val_df.dropna(
    subset=["Lag_52"]
)

y_train = train_df.pop("Weekly_Sales")
y_val = val_df.pop("Weekly_Sales")

X_train = train_df.drop(columns=["Date"])
X_val = val_df.drop(columns=["Date"])

In [ ]:
import numpy as np

from lightgbm import LGBMRegressor

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error



class MarkdownFiller:

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = X.copy()

        markdown_cols = [
            "MarkDown1",
            "MarkDown2",
            "MarkDown3",
            "MarkDown4",
            "MarkDown5"
        ]

        for col in markdown_cols:
            if col in X.columns:
                X[col] = X[col].fillna(0)

        return X



preprocessor = ColumnTransformer(

    transformers=[

        (

            "categorical",

            OneHotEncoder(
                handle_unknown="ignore"
            ),

            make_column_selector(
                dtype_include=[
                    "object",
                    "category",
                    "bool"
                ]
            )

        ),

        (

            "numerical",

            SimpleImputer(
                strategy="median"
            ),

            make_column_selector(
                dtype_include=np.number
            )

        )

    ]

)



lgbm = LGBMRegressor(

    objective="regression",

    random_state=42,

    n_jobs=-1

)


pipeline = Pipeline(

    steps=[

        (
            "markdown_fill",
            MarkdownFiller()
        ),

        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            lgbm
        )

    ]

)



param_grid = {

    "model__n_estimators": [
        200,
        300,
        500,
        700,
        1000
    ],

    "model__learning_rate": [
        0.01,
        0.02,
        0.03,
        0.05
    ],

    "model__max_depth": [
        3,
        5,
        6,
        7,
        10
    ],

    "model__num_leaves": [
        8,
        16,
        24,
        31
    ],

    "model__min_child_samples": [
        20,
        50,
        100
    ],

    "model__subsample": [
        0.5,
        0.6,
        0.7,
        0.8
    ],

    "model__colsample_bytree": [
        0.6,
        0.8,
        1.0
    ],

    "model__reg_alpha": [
        0,
        0.01,
        0.1,
    ],

    "model__reg_lambda": [
        5,
        10,
        20
    ]

}



tscv = TimeSeriesSplit(
    n_splits=3
)


random_search = RandomizedSearchCV(

    estimator=pipeline,

    param_distributions=param_grid,

    n_iter=50,

    scoring="neg_mean_absolute_error",

    cv=tscv,

    verbose=2,

    random_state=42,

    n_jobs=-1

)



random_search.fit(

    train_df,

    y_train

)


best_model = random_search.best_estimator_


print(random_search.best_params_)



train_pred = best_model.predict(
    train_df
)

val_pred = best_model.predict(
    val_df
)



train_mae = mean_absolute_error(
    y_train,
    train_pred
)

val_mae = mean_absolute_error(
    y_val,
    val_pred
)


train_weights = np.where(
    train_df["IsHoliday"],
    5,
    1
)

val_weights = np.where(
    val_df["IsHoliday"],
    5,
    1
)


train_wmae = (
    np.sum(
        train_weights *
        np.abs(y_train - train_pred)
    )
    /
    np.sum(train_weights)
)


val_wmae = (
    np.sum(
        val_weights *
        np.abs(y_val - val_pred)
    )
    /
    np.sum(val_weights)
)


print("Train MAE:", train_mae)
print("Train WMAE:", train_wmae)

print("Validation MAE:", val_mae)
print("Validation WMAE:", val_wmae)

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009843 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1558
[LightGBM] [Info] Number of data points in the train set: 73092, number of used features: 18
[LightGBM] [Info] Start training from score 16288.944046
{'model__subsample': 0.8, 'model__reg_lambda': 10, 'model__reg_alpha': 0.1, 'model__num_leaves': 31, 'model__n_estimators': 300, 'model__min_child_samples': 20, 'model__max_depth': 10, 'model__learning_rate': 0.05, 'model__colsample_bytree': 1.0}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Train MAE: 1450.4542801798707
Train WMAE: 1464.5073129410594
Validation MAE: 2031.2013782163524
Validation WMAE: 2180.371703292339


In [ ]:
random_search.best_params_

{'model__subsample': 0.6,
 'model__reg_lambda': 10,
 'model__reg_alpha': 0,
 'model__num_leaves': 31,
 'model__n_estimators': 300,
 'model__min_child_samples': 20,
 'model__max_depth': 7,
 'model__learning_rate': 0.03,
 'model__colsample_bytree': 1.0}

In [ ]:
mlflow.set_experiment("LGBM_Experiment")
with mlflow.start_run(
    run_name="LGBM_run_1"
):

    mlflow.log_params(
        random_search.best_params_
    )

    mlflow.log_metrics({
        "train_MAE": train_mae,
        "train_WMAE": train_wmae,
        "val_MAE": val_mae,
        "val_WMAE": val_wmae
    })



🏃 View run LGBM_run_1 at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/2fb4487ff68d439284de19e3d5121d86
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/3


In [ ]:
from lightgbm import LGBMRegressor

lgbm_model = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1,

    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=16,
    min_child_samples=50,
    subsample=0.7,
    colsample_bytree=1.0,
    reg_alpha=0.1,
    reg_lambda=5
)
pipeline = Pipeline(
    steps=[
        (
            "markdown_fill",
            MarkdownFiller()
        ),
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            lgbm_model
        )
    ]
)
pipeline.fit(
    X_train,
    y_train
)

best_model = pipeline
train_pred = best_model.predict(X_train)

val_pred = best_model.predict(X_val)
from sklearn.metrics import mean_absolute_error

train_mae = mean_absolute_error(
    y_train,
    train_pred
)

val_mae = mean_absolute_error(
    y_val,
    val_pred
)

train_weights = np.where(
    X_train["IsHoliday"],
    5,
    1
)

val_weights = np.where(
    X_val["IsHoliday"],
    5,
    1
)

train_wmae = (
    np.sum(train_weights * np.abs(y_train - train_pred))
    / np.sum(train_weights)
)

val_wmae = (
    np.sum(val_weights * np.abs(y_val - val_pred))
    / np.sum(val_weights)
)

print("Train MAE:", train_mae)
print("Train WMAE:", train_wmae)
print("Validation MAE:", val_mae)
print("Validation WMAE:", val_wmae)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010481 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1558
[LightGBM] [Info] Number of data points in the train set: 73092, number of used features: 18
[LightGBM] [Info] Start training from score 16288.944046
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Train MAE: 1276.3874655503857
Train WMAE: 1285.0427029470186
Validation MAE: 2130.7311908494876
Validation WMAE: 2314.1316780720385


In [ ]:
mlflow.set_experiment("LGBM_Experiment")
with mlflow.start_run(
    run_name="LGBM_run_2_with last final params"
):

    mlflow.log_params(
        {
            "model": "LightGBM",
            "n_estimators": 2000,
            "learning_rate": 0.05,
            "max_depth": 6,
            "num_leaves": 16,
            "min_child_samples": 50,
            "subsample": 0.7,
            "colsample_bytree": 1.0,
            "reg_alpha": 0.1,
            "reg_lambda": 5,
            "features": "Lag_52_only"
        }
    )

    mlflow.log_metrics({
        "train_MAE": train_mae,
        "train_WMAE": train_wmae,
        "val_MAE": val_mae,
        "val_WMAE": val_wmae
    })



🏃 View run LGBM_run_2_with last final params at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/2179a07e4b0e48d9b5dad460f7285e8d
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/3
